# 🔥 ClusterOps: Thermal GPU Balancer — RL via Behavioral Cloning

**To solve the negative rewards and Colab crashes, we have upgraded the training pipeline!**

Instead of an unstable interactive loop, we now use **Offline Reinforcement Learning (Behavioral Cloning)**:
1. An expert heuristic agent plays the game and generates a "Golden Dataset" of perfect thermal management.
2. We use `SFTTrainer` and `Unsloth` to fine-tune the LLM to internalize these perfect strategies.
3. The result is a highly stable, fast-training model that gets **positive rewards**!

---

## 1. Install Dependencies

In [ ]:
# Do NOT use xformers<0.0.27 on modern Colab (Py 3.12 + torch 2.4+): pip builds from source and fails.
# Let Unsloth pull a compatible `trl`; do not pin `trl<0.9` here (it fights Unsloth + new Transformers).
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers peft accelerate bitsandbytes triton -q
!pip install fastapi uvicorn requests pydantic "openenv-core[core]>=0.2.2" -q

import importlib.util as _iu
assert _iu.find_spec("unsloth") and _iu.find_spec("xformers") and _iu.find_spec("trl"), "Install failed — scroll up for pip errors"
print("✅ Dependencies installed (unsloth, xformers, trl OK)")

## 2. Start ClusterOps Environment Server

In [ ]:
import subprocess, time, requests, os, sys, json

REPO_URL = "https://github.com/Sushmit-Biswas/thermal-gpu-balancer.git"
REPO_DIR = "/content/clusterops_repo"
SERVER_LOG = "/content/clusterops_server.log"

os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True)

os.chdir("/content")
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)

print("Cloning repo...")
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)


def _stop_server(proc):
    if proc is None:
        return
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)


def start_server():
    global server_proc, server_log_handle

    if "server_proc" in globals():
        _stop_server(server_proc)

    if "server_log_handle" in globals() and server_log_handle and not server_log_handle.closed:
        server_log_handle.close()

    # Use file redirection instead of PIPE to avoid uvicorn log-buffer deadlocks.
    server_log_handle = open(SERVER_LOG, "w")
    server_proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "server.app:app",
            "--host",
            "0.0.0.0",
            "--port",
            "8000",
            "--log-level",
            "warning",
        ],
        stdout=server_log_handle,
        stderr=subprocess.STDOUT,
    )

    for _ in range(30):
        time.sleep(1)
        try:
            if requests.get("http://localhost:8000/health", timeout=2).ok:
                print("✅ Server online")
                return
        except Exception:
            pass

    _stop_server(server_proc)
    tail = ""
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG, "r", errors="ignore") as f:
            tail = "".join(f.readlines()[-30:])
    raise RuntimeError(f"Server failed to start. Recent logs:\n{tail}")


start_server()
print(f"Server log file: {SERVER_LOG}")

## 3. Generate Golden Training Data (The Secret to Positive Rewards!)

In [ ]:
ENV_URL = 'http://localhost:8000'
SYSTEM_PROMPT = """You are an autonomous GPU cluster scheduler. Output ONE JSON action only. {"action_type": "allocate", "job_id": "job_1", "node_id": 2}"""

training_data = []

def collect_expert_trajectory():
    data = requests.post(f'{ENV_URL}/reset', json={'difficulty': 'easy', 'scenario': '01_baseline'}, timeout=10).json()
    obs = data.get('observation', {})
    trajectory = []
    total_reward = 0.0

    while not data.get('done', False):
        nodes = obs.get('gpu_nodes', [])
        queue = obs.get('job_queue', [])
        
        action = {'action_type': 'wait'}
        for n in nodes:
            if n['status'] == 'busy' and n['temperature'] >= 92.0:
                action = {'action_type': 'evict', 'node_id': n['id']}
                break
        
        if action['action_type'] == 'wait' and queue:
            idle_nodes = [n for n in nodes if n['status'] == 'idle']
            if idle_nodes:
                best_node = min(idle_nodes, key=lambda x: x['temperature'])
                action = {'action_type': 'allocate', 'job_id': queue[0]['id'], 'node_id': best_node['id']}
        
        prompt = f"Step {data.get('metadata', {}).get('step')}: {obs}"
        response = json.dumps(action)
        trajectory.append({"instruction": prompt, "output": response})
        
        data = requests.post(f'{ENV_URL}/step', json=action, timeout=10).json()
        obs = data.get('observation', {})
        total_reward += data.get('reward', 0)
        
    return trajectory, total_reward

print("Gathering perfect data...")
for ep in range(5):
    traj, reward = collect_expert_trajectory()
    training_data.extend(traj)
    print(f"  Expert Episode {ep+1}: Reward = {reward:.1f}")

print(f"✅ Collected {len(training_data)} perfect training steps!")


## 4. Load Model & Prepare Dataset

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset

# Qwen2.5 1.5B is still Colab-friendly, with better policy quality than the 0.5B variant.
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)


def format_prompts(examples):
    """Use the tokenizer's chat template so SFT strings match Qwen (or any) Instruct format."""
    texts = []
    for instruction, output in zip(examples['instruction'], examples['output']):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': instruction},
            {'role': 'assistant', 'content': output},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}


dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_prompts, batched=True)
print(f"✅ Model ({MODEL_NAME}) and Dataset Ready!")

## 5. Fine-Tune the LLM (Fast SFT Training)

In [ ]:
import inspect

import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# TRL >= ~0.16 + recent Transformers: use `processing_class` (not `tokenizer`) and put
# dataset_* / max_seq_length on `SFTConfig`. Older TRL keeps the legacy kwargs on SFTTrainer.
_train_sig = inspect.signature(SFTTrainer.__init__)
_common_args = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
)

if "processing_class" in _train_sig.parameters:
    from trl import SFTConfig

    args = SFTConfig(
        **_common_args,
        max_seq_length=1024,
        dataset_text_field="text",
        dataset_num_proc=2,
    )
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
else:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=1024,
        dataset_num_proc=2,
        args=TrainingArguments(**_common_args),
    )

print(
    "Using SFTConfig + processing_class (TRL/Transformers new API)"
    if "processing_class" in _train_sig.parameters
    else "Using tokenizer= + TrainingArguments (TRL legacy API)"
)
print("🚀 Starting Training... This will show a nice progress bar and won't crash!")
trainer_stats = trainer.train()
print("✅ Training Complete!")

## 6. Evaluate the Trained Model (Fast)

Lean evaluation tuned for a **<5 min total Colab run**:
- Single scenario: `01_baseline` at **easy** (≈60 env steps per episode)
- 2 episodes per policy (Naive / Expert / Trained)
- Short JSON generations (`max_new_tokens=24`)
- Per-episode prints + auto server-restart on timeouts

Outputs `naive_rewards`, `expert_rewards`, `trained_rewards` — consumed directly by Cell 7.

In [ ]:
import logging as _py_logging
import time
import warnings
import json

import requests
import torch
import transformers
from requests.exceptions import ReadTimeout, ConnectionError as _ReqConnErr

# Silence per-call generate logs / repeated warning spam during eval.
transformers.utils.logging.set_verbosity_error()
_py_logging.getLogger("transformers").setLevel(_py_logging.ERROR)
_py_logging.getLogger("transformers.generation").setLevel(_py_logging.ERROR)
warnings.filterwarnings("ignore", message=r"Both `max_new_tokens`")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"transformers\\.modeling_attn_mask_utils")
warnings.filterwarnings("ignore", message=r"Passing `generation_config` together with")

# Lean eval — designed to finish in <1 minute on T4 after training.
SCENARIO = "01_baseline"
DIFFICULTY = "easy"
N_EVAL_EPISODES = 2
HTTP_TIMEOUT = 15
HTTP_ATTEMPTS = 3
MAX_EPISODE_STEPS = 80   # easy difficulty already caps at 60; small buffer.

_RESET_BODY = {"difficulty": DIFFICULTY, "scenario": SCENARIO}
_STEP_FALLBACK = {"observation": {}, "metadata": {}, "reward": 0.0, "done": True}


def _wait_for_server(max_wait_s: int = 15) -> bool:
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        try:
            if requests.get(f"{ENV_URL}/health", timeout=2).ok:
                return True
        except Exception:
            pass
        time.sleep(0.5)
    return False


def _restart_server_if_needed():
    if "start_server" not in globals():
        raise RuntimeError("start_server() is missing. Rerun Cell 4 to launch the API server.")
    proc = globals().get("server_proc")
    if proc is None or proc.poll() is not None:
        print("    [warn] API server is down. Restarting...")
        start_server()
        return
    if not _wait_for_server(max_wait_s=2):
        print("    [warn] API health check failed. Restarting server...")
        start_server()


def _post(path, payload, attempts=HTTP_ATTEMPTS, timeout=HTTP_TIMEOUT, fallback=None):
    last_err = None
    for k in range(attempts):
        _restart_server_if_needed()
        try:
            resp = requests.post(f"{ENV_URL}{path}", json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except (ReadTimeout, _ReqConnErr, requests.HTTPError, ValueError) as e:
            last_err = e
            wait = min(2 ** k, 5)
            print(f"    [warn] POST {path} failed (try {k+1}/{attempts}): {type(e).__name__}; retry in {wait}s")
            time.sleep(wait)
    if fallback is not None:
        print(f"    [warn] POST {path} exhausted retries; using fallback ({last_err})")
        return fallback
    raise RuntimeError(f"POST {path} failed after {attempts} retries: {last_err}")


def run_naive_episode():
    """Always-wait baseline."""
    data = _post("/reset", _RESET_BODY)
    total = 0.0
    for _ in range(MAX_EPISODE_STEPS):
        if data.get("done", False):
            break
        data = _post("/step", {"action_type": "wait"}, fallback=_STEP_FALLBACK)
        total += data.get("reward", 0.0)
    return total


def run_expert_episode():
    """Heuristic teacher (oracle upper bound)."""
    data = _post("/reset", _RESET_BODY)
    obs = data.get("observation", {})
    total = 0.0
    for _ in range(MAX_EPISODE_STEPS):
        if data.get("done", False):
            break
        nodes = obs.get("gpu_nodes", [])
        queue = obs.get("job_queue", [])
        action = {"action_type": "wait"}
        for n in nodes:
            if n["status"] == "busy" and n["temperature"] >= 92.0:
                action = {"action_type": "evict", "node_id": n["id"]}
                break
        if action["action_type"] == "wait" and queue:
            idle = [n for n in nodes if n["status"] == "idle"]
            if idle:
                best = min(idle, key=lambda x: x["temperature"])
                action = {"action_type": "allocate", "job_id": queue[0]["id"], "node_id": best["id"]}
        data = _post("/step", action, fallback=_STEP_FALLBACK)
        obs = data.get("observation", {})
        total += data.get("reward", 0.0)
    return total


def _parse_action_json(text):
    t = text.strip()
    if "```" in t:
        t = t.replace("```json", "").replace("```", "")
    i, j = t.find("{"), t.rfind("}")
    if i < 0 or j < i:
        return {"action_type": "wait"}
    try:
        a = json.loads(t[i:j + 1])
        if isinstance(a, dict) and a.get("action_type"):
            return a
    except Exception:
        pass
    return {"action_type": "wait"}


_pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
_GEN_KW = dict(
    max_new_tokens=24,            # JSON action is short — keep gen tight for speed
    do_sample=False,
    pad_token_id=_pad_id,
    eos_token_id=tokenizer.eos_token_id,
)


@torch.no_grad()
def run_trained_episode():
    data = _post("/reset", _RESET_BODY)
    total = 0.0
    for _ in range(MAX_EPISODE_STEPS):
        if data.get("done", False):
            break
        obs, meta = data.get("observation", {}), data.get("metadata", {})
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Step {meta.get('step')}: {obs}"},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, **_GEN_KW)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        action = _parse_action_json(text)
        data = _post("/step", action, fallback=_STEP_FALLBACK)
        total += data.get("reward", 0.0)
    return total


def _run(label, runner, n):
    rewards = []
    for i in range(n):
        t0 = time.time()
        try:
            r = runner()
        except Exception as e:
            print(f"  [{label}] Episode {i+1}: FAILED ({type(e).__name__}: {e}); recording 0")
            _wait_for_server(max_wait_s=5)
            r = 0.0
        rewards.append(r)
        print(f"  {label} Episode {i+1}: Reward = {r:.1f}   ({time.time()-t0:.1f}s)")
    return rewards


print(f"=== Eval · {SCENARIO} @ {DIFFICULTY} · {N_EVAL_EPISODES} episodes ===")

print("--- Naive baseline (always wait) ---")
naive_rewards = _run("Naive", run_naive_episode, N_EVAL_EPISODES)

print("--- Expert teacher (oracle) ---")
expert_rewards = _run("Expert", run_expert_episode, N_EVAL_EPISODES)

print("--- Trained LLM (BC) ---")
trained_rewards = _run("Trained", run_trained_episode, N_EVAL_EPISODES)

print(
    f"\nMeans · Naive={sum(naive_rewards)/len(naive_rewards):.1f}  "
    f"Expert={sum(expert_rewards)/len(expert_rewards):.1f}  "
    f"Trained={sum(trained_rewards)/len(trained_rewards):.1f}"
)

## 7. Training visualizations (saved to `assets/`)

Loss: per-step logs plus a light moving average so the curve reads smoothly. Rewards: **naive (always wait)** vs **trained BC** (fair “did we learn anything?”), with **expert** as a dotted upper bound — not labeled as “untrained.” Files go under `assets/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ASSETS_DIR = os.path.join(REPO_DIR, 'assets')
os.makedirs(ASSETS_DIR, exist_ok=True)

# Sleek dark palette
plt.style.use('dark_background')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titleweight': 'bold',
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'axes.edgecolor': '#2a2f3a',
    'axes.linewidth': 1.0,
    'axes.grid': True,
    'grid.color': '#2a2f3a',
    'grid.linestyle': '--',
    'grid.alpha': 0.35,
    'legend.frameon': True,
    'legend.facecolor': '#10141c',
    'legend.edgecolor': '#2a2f3a',
    'legend.fontsize': 10,
})

success_color = '#2cb67d'
trained_glow = '#3ddc97'
danger_color = '#ef4565'
loss_glow    = '#ff8ba3'
muted_color  = '#9aa1ad'
gold_color   = '#f5a623'
face         = '#0d1117'


def _despine(ax):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color('#2a2f3a')


def moving_average(y, window):
    y = np.asarray(y, dtype=float)
    if len(y) < 3:
        return y.copy()
    w = min(window, len(y) if len(y) % 2 == 1 else len(y) - 1)
    if w < 3:
        w = 3
    if w % 2 == 0:
        w -= 1
    kernel = np.ones(w) / w
    return np.convolve(y, kernel, mode='same')


paths_saved = []

# ── 1. Loss: raw + smoothed (sleek convergence curve) ────────────────────────
history = getattr(trainer.state, 'log_history', []) or []
loss_raw = [x['loss'] for x in history if 'loss' in x]
steps    = [x['step'] for x in history if 'loss' in x]

if loss_raw:
    loss_smooth = moving_average(loss_raw, window=7)

    fig, ax = plt.subplots(figsize=(10, 6), facecolor=face)
    ax.set_facecolor(face)
    ax.plot(steps, loss_raw, color=danger_color, linewidth=1.2, alpha=0.30, label='Raw loss (per step)')
    ax.plot(steps, loss_smooth, color=loss_glow, linewidth=3.2, label='Smoothed (moving avg)')
    ax.fill_between(steps, loss_smooth, alpha=0.10, color=danger_color)

    start, end = float(loss_smooth[0]), float(loss_smooth[-1])
    delta_pct = (start - end) / start * 100.0 if start > 0 else 0.0
    ax.annotate(
        f'↓ {delta_pct:.1f}% loss reduction',
        xy=(steps[-1], end), xytext=(-12, 18), textcoords='offset points',
        ha='right', va='bottom', fontsize=10, color=trained_glow, fontweight='bold',
    )

    ax.set_title('ClusterOps · SFT Training Convergence', pad=18)
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Loss')
    ax.legend(loc='upper right')
    _despine(ax)
    fig.tight_layout()

    loss_path = os.path.join(ASSETS_DIR, 'loss_curve.png')
    fig.savefig(loss_path, dpi=300, bbox_inches='tight', facecolor=face)
    plt.show()
    plt.close(fig)
    paths_saved.append(loss_path)
else:
    print('[warn] No loss history found — skipping loss curve.')

# ── 2. Episode Return: naive vs trained, expert as upper bound ───────────────
naive_arr   = np.asarray(naive_rewards,   dtype=float)
trained_arr = np.asarray(trained_rewards, dtype=float)
expert_arr  = np.asarray(expert_rewards,  dtype=float)

if trained_arr.size:
    ep_x = np.arange(1, trained_arr.size + 1)

    fig, ax = plt.subplots(figsize=(10, 6), facecolor=face)
    ax.set_facecolor(face)

    ax.plot(ep_x, naive_arr,   label='Naive (always wait)',
            linestyle='--', marker='o', color=muted_color, alpha=0.85, linewidth=2, markersize=7)
    ax.plot(ep_x, trained_arr, label='Trained LLM (BC)',
            color=success_color, marker='s', linewidth=3.5, markersize=9,
            markeredgecolor=trained_glow, markeredgewidth=1.2)
    ax.plot(ep_x, expert_arr,  label='Expert heuristic (teacher)',
            linestyle=':', marker='^', color=gold_color, linewidth=2, markersize=8, alpha=0.95)

    ax.fill_between(
        ep_x,
        np.minimum(naive_arr, trained_arr),
        np.maximum(naive_arr, trained_arr),
        color=success_color, alpha=0.10, label='_nolegend_',
    )

    naive_mean   = float(naive_arr.mean())
    trained_mean = float(trained_arr.mean())
    expert_mean  = float(expert_arr.mean())
    lift = trained_mean - naive_mean
    ax.text(
        0.98, 0.05,
        f'Trained mean: {trained_mean:.1f}   ·   Naive mean: {naive_mean:.1f}   ·   Lift: {lift:+.1f}',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=10, color='#cfd3dc',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#10141c', edgecolor='#2a2f3a'),
    )

    ax.set_title('Episode Return · Naive vs Trained (Expert = Upper Bound)', pad=18)
    ax.set_xlabel('Evaluation Episode')
    ax.set_ylabel('Total Cumulative Reward')
    ax.set_xticks(ep_x)
    ax.legend(loc='best')
    _despine(ax)
    fig.tight_layout()

    reward_path = os.path.join(ASSETS_DIR, 'reward_curve.png')
    fig.savefig(reward_path, dpi=300, bbox_inches='tight', facecolor=face)
    plt.show()
    plt.close(fig)
    paths_saved.append(reward_path)
else:
    print('[warn] No evaluation rewards found — skipping reward curve.')

if paths_saved:
    print('Saved:\n  ' + '\n  '.join(paths_saved))
    try:
        from google.colab import files
        for p in paths_saved:
            files.download(p)
    except Exception:
        pass